In [6]:
from playwright.sync_api import sync_playwright
import pandas as pd
import time
import random

MODELOS = {
    "toyota-etios": "Toyota Etios",
    "toyota-corolla": "Toyota Corolla",
    "toyota-yaris": "Toyota Yaris",
    "chevrolet-onix-plus": "Chevrolet Onix Plus",
    "chevrolet-prisma": "Chevrolet Prisma",
    "fiat-cronos": "Fiat Cronos",
    "renault-logan": "Renault Logan",
    "volkswagen-virtus": "VW Virtus",
    "volkswagen-voyage": "VW Voyage",
    "nissan-versa": "Nissan Versa",
    "fiat-siena": "Fiat Siena",
    "volkswagen-polo": "VW Polo",
    "volkswagen-gol": "VW Gol",
    "ford-fiesta": "Ford Fiesta",
    "ford-ka": "Ford Ka",
    "peugeot-208": "Peugeot 208",
}

def scrape_modelo(page, slug, nombre, paginas=5):
    resultados = []

    for pagina in range(1, paginas + 1):
        offset = (pagina - 1) * 48
        url = f"https://autos.mercadolibre.com.ar/{slug}_Desde_{offset + 1}_NoIndex_True"

        try:
            page.goto(url, timeout=30000)
            page.wait_for_selector("li.ui-search-layout__item", timeout=15000)

            publicaciones = page.query_selector_all("li.ui-search-layout__item")

            for pub in publicaciones:
                try:
                    titulo = pub.query_selector("h2").inner_text()

                    precio_el = pub.query_selector("span.andes-money-amount__fraction")
                    precio_txt = precio_el.inner_text() if precio_el else None
                    precio = int(precio_txt.replace(".", "")) if precio_txt else None

                    atributos = pub.query_selector_all("li.ui-search-card-attributes__attribute")
                    año = atributos[0].inner_text() if len(atributos) > 0 else None
                    km = atributos[1].inner_text() if len(atributos) > 1 else None

                    resultados.append({
                        "modelo": nombre,
                        "titulo": titulo,
                        "precio_ars": precio,
                        "año": año,
                        "km": km
                    })
                except:
                    continue

        except Exception as e:
            print(f"  ⚠ Error en {nombre} página {pagina}: {e}")

        time.sleep(random.uniform(2.0, 4.0))

    return resultados


def main():
    todos = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        for slug, nombre in MODELOS.items():
            print(f"Scrapeando {nombre}...")
            datos = scrape_modelo(page, slug, nombre)
            todos.extend(datos)
            print(f"  → {len(datos)} publicaciones encontradas")

        browser.close()

    df = pd.DataFrame(todos)
    df.to_csv("../data/raw/precios_autos_raw.csv", index=False)
    print(f"\nTotal: {len(df)} registros guardados en data/raw/precios_autos_raw.csv")
    return df

df = main()
df.head(10)

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.